# Exploración Inicial de Datos — Admisiones de Posgrado

**Autor:** SCO

**Fecha:** 2026-08-19

**Descripción:**
Notebook del Issue #2 "Exploración inicial de datos". Describe el dataset RAW
(`data/01_raw/Admission_Predict.csv`), unifica la representación de los valores
nulos, convierte cada columna a su tipo correcto y guarda el resultado como
Parquet en `data/02_intermediate/Admission_Predict.parquet`.

No imputa ni elimina registros por valores nulos, no crea features nuevas y no
realiza EDA, tratamiento de outliers, escalado, encoding ni modelamiento
(etapas posteriores del curso).


## 📚 Import libraries

In [1]:
# base libraries for data science
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa

print(pd.__version__)

3.0.5


## 💾 Load data

Lectura **reproducible** e **inmutable** del RAW: se localiza la raíz del
repositorio (buscando `pyproject.toml`) y se lee
`data/01_raw/Admission_Predict.csv` sin modificarlo.


In [2]:
def find_repo_root(start: Path) -> Path:
    """Localiza la raíz del repositorio subiendo hasta encontrar `pyproject.toml`."""
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("No se encontró la raíz del repositorio (pyproject.toml).")


ROOT = find_repo_root(Path.cwd())

DATA_DIR = ROOT / "data"
RAW_PATH = DATA_DIR / "01_raw" / "Admission_Predict.csv"
INTERMEDIATE_DIR = DATA_DIR / "02_intermediate"
OUTPUT_PATH = INTERMEDIATE_DIR / "Admission_Predict.parquet"

# Lectura del RAW (inmutable): pandas convierte las celdas vacías en NaN por defecto.
admission_df = pd.read_csv(RAW_PATH)

admission_df.head()

,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
0,337.0,118.0,4.0,4.5,4.5,9.65,1.0,0.92
1,324.0,107.0,4.0,4.0,4.5,8.87,1.0,0.76
2,316.0,104.0,3.0,3.0,3.5,8.00,1.0,0.72
3,322.0,110.0,3.0,3.5,2.5,8.67,1.0,0.80
4,314.0,103.0,2.0,2.0,3.0,8.21,0.0,0.65


## 👷 Data preparation or Feature Engineering

### 📊 Descripción de los datos

Dimensiones, columnas, registros y esquema/tipos actuales del RAW.


In [3]:
print(f"Shape (filas, columnas): {admission_df.shape}")
print(f"Columnas: {list(admission_df.columns)}")
print()
admission_df.info()

Shape (filas, columnas): (623, 8)
Columnas: ['GRE Score', 'TOEFL Score', 'University Rating', 'SOP', 'LOR ', 'CGPA', 'Research', 'Chance of Admit ']

<class 'pandas.DataFrame'>
RangeIndex: 623 entries, 0 to 622
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   GRE Score          612 non-null    float64
 1   TOEFL Score        604 non-null    float64
 2   University Rating  617 non-null    float64
 3   SOP                607 non-null    float64
 4   LOR                615 non-null    float64
 5   CGPA               621 non-null    float64
 6   Research           594 non-null    float64
 7   Chance of Admit    623 non-null    float64
dtypes: float64(8)
memory usage: 39.1 KB


In [4]:
admission_df.sample(10, random_state=42)

,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
249,321.0,111.0,3.0,3.5,4.0,8.83,1.0,0.77
558,326.0,112.0,3.0,3.5,3.0,9.05,NaN,0.74
174,321.0,111.0,4.0,4.0,4.0,8.97,1.0,0.87
280,311.0,102.0,3.0,4.5,4.0,8.64,1.0,0.68
110,305.0,108.0,5.0,3.0,3.0,8.48,0.0,0.61
244,314.0,107.0,2.0,2.5,4.0,8.56,0.0,0.63
228,318.0,112.0,3.0,4.0,3.5,8.67,0.0,0.71
227,312.0,110.0,2.0,3.5,3.0,8.53,0.0,0.64
465,316.0,100.0,2.0,1.5,3.0,8.16,1.0,0.71
148,339.0,116.0,4.0,4.0,3.5,9.80,1.0,0.96


### Valores nulos

Se identifica cómo están representados los valores faltantes en el RAW y se
normalizan a una única representación (`NaN`).

- **Lectura estándar (`read_csv`)**: pandas convierte automáticamente las
  celdas vacías a `NaN` al leer el CSV (es la lectura que usa `admission_df`).
- **Inspección de la representación original**: para ver *cómo* están
  codificados los faltantes en el archivo RAW se lee con
  `keep_default_na=False`, que conserva los valores tal cual (la celda
  siguiente muestra que son cadenas vacías `''`, sin otros marcadores).
- **Normalización**: toda celda vacía o de solo espacios en blanco se
  convierte a `NaN` (`replace(r"^\s*$", np.nan, regex=True)`).

**No se imputan valores ni se eliminan filas**: los nulos se conservan para
su tratamiento en etapas posteriores.


In [5]:
# Lectura SIN conversión automática para evidenciar la representación cruda del nulo.
raw_naive = pd.read_csv(RAW_PATH, keep_default_na=False)
str_df = raw_naive.astype(str)

# Celdas vacías (cadena '') — representación de nulo en el RAW.
empty_counts = (str_df == "").sum()

# Celdas con solo espacios en blanco (posible segunda representación).
whitespace_only = str_df.map(lambda x: x != "" and x.strip() == "").sum()

print("Celdas vacías ('') por columna:")
print(empty_counts[empty_counts > 0])
print()
print("Celdas con solo espacios en blanco (total):", int(whitespace_only.sum()))

Celdas vacías ('') por columna:
GRE Score            11
TOEFL Score          19
University Rating     6
SOP                  16
LOR                   8
CGPA                  2
Research             29
dtype: int64

Celdas con solo espacios en blanco (total): 0


In [6]:
# Normalización: toda celda vacía o de solo espacios en blanco pasa a NaN.
# NO se imputan valores ni se eliminan registros.
admission_df = admission_df.replace(r"^\s*$", np.nan, regex=True)

print("Valores nulos (NaN) tras normalizar:")
print(admission_df.isna().sum())
print()
print("Total de nulos:", int(admission_df.isna().sum().sum()))

Valores nulos (NaN) tras normalizar:
GRE Score            11
TOEFL Score          19
University Rating     6
SOP                  16
LOR                   8
CGPA                  2
Research             29
Chance of Admit       0
dtype: int64

Total de nulos: 91


### Conversión de tipos

Cada columna se convierte a su tipo esperado:

| Columna | Tipo esperado | Justificación |
| --- | --- | --- |
| `GRE Score` | entero (`Int64`) | puntaje discreto 290–340; solo valores enteros; con nulos |
| `TOEFL Score` | entero (`Int64`) | puntaje discreto 92–120; solo valores enteros; con nulos |
| `University Rating` | entero (`Int64`) | ordinal 1–5; solo valores enteros; con nulos |
| `SOP` | float (`float64`) | escala 1.0–5.0 con pasos de 0.5; con nulos |
| `LOR` | float (`float64`) | escala 1.0–5.0 con pasos de 0.5; con nulos |
| `CGPA` | float (`float64`) | continua 6.8–9.92; con nulos |
| `Research` | entero (`Int64`) | binaria 0/1; solo valores enteros; con nulos |
| `Chance of Admit` | float (`float64`) | target continua 0.34–0.97; sin nulos |

Se usa el dtype nullable `Int64` para las enteras porque contienen nulos;
las continuas usan `float64`, cuyo nulo es `NaN`.

Además se normalizan los nombres de columna quitando los espacios en blanco al
inicio y al final (`LOR ` → `LOR`, `Chance of Admit ` → `Chance of Admit`), una
inconsistencia básica del RAW.


In [7]:
# Normalización de nombres de columnas: quitar espacios en blanco iniciales/finales.
# ('LOR ' y 'Chance of Admit ' traen un espacio final en el RAW.)
admission_df.columns = admission_df.columns.str.strip()
print("Columnas normalizadas:", list(admission_df.columns))
print()

# Conversión a tipos correctos.
# Enteras discretas con nulos -> dtype nullable Int64 (los nulos pasan a pd.NA).
int_cols = ["GRE Score", "TOEFL Score", "University Rating", "Research"]
admission_df[int_cols] = admission_df[int_cols].astype("Int64")

# Continuas / con decimales -> float64 (los nulos se conservan como NaN).
float_cols = ["SOP", "LOR", "CGPA", "Chance of Admit"]
admission_df[float_cols] = admission_df[float_cols].astype("float64")

admission_df.info()

Columnas normalizadas: ['GRE Score', 'TOEFL Score', 'University Rating', 'SOP', 'LOR', 'CGPA', 'Research', 'Chance of Admit']

<class 'pandas.DataFrame'>
RangeIndex: 623 entries, 0 to 622
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   GRE Score          612 non-null    Int64  
 1   TOEFL Score        604 non-null    Int64  
 2   University Rating  617 non-null    Int64  
 3   SOP                607 non-null    float64
 4   LOR                615 non-null    float64
 5   CGPA               621 non-null    float64
 6   Research           594 non-null    Int64  
 7   Chance of Admit    623 non-null    float64
dtypes: Int64(4), float64(4)
memory usage: 41.5 KB


### 💾 Guardar dataset con tipos de datos

Se guarda el DataFrame tipado en formato Parquet (con el esquema pyarrow
derivado de los tipos) en `data/02_intermediate/Admission_Predict.parquet`.


In [8]:
# Esquema pyarrow derivado del DataFrame ya tipado.
schema = pa.Table.from_pandas(admission_df).schema
print("Esquema pyarrow:")
print(schema)
print()

INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

admission_df.to_parquet(OUTPUT_PATH, index=False, schema=schema)
print("Guardado:", OUTPUT_PATH)

Esquema pyarrow:
GRE Score: int64
TOEFL Score: int64
University Rating: int64
SOP: double
LOR: double
CGPA: double
Research: int64
Chance of Admit: double
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 1208

Guardado: /home/elkiruvi/Proyecto-Admisiones/data/02_intermediate/Admission_Predict.parquet


### ✅ Validación

Se relee el Parquet y se validan tipos, nulos y dimensiones contra el DataFrame
intermedio.


In [9]:
# Lectura de vuelta del Parquet para validar el resultado.
df_parquet = pd.read_parquet(OUTPUT_PATH)

print("Shape al releer:", df_parquet.shape)
print("Columnas al releer:", list(df_parquet.columns))
print()
print("Tipos al releer:")
print(df_parquet.dtypes)
print()
print("Nulos al releer:")
print(df_parquet.isna().sum())

# Validaciones formales
assert df_parquet.shape == admission_df.shape
assert list(df_parquet.columns) == list(admission_df.columns)
assert df_parquet.dtypes.to_dict() == admission_df.dtypes.to_dict()
assert df_parquet.isna().sum().to_dict() == admission_df.isna().sum().to_dict()

print()
print("Validación OK: dimensiones, columnas, tipos y nulos coinciden con el DataFrame intermedio.")

Shape al releer: (623, 8)
Columnas al releer: ['GRE Score', 'TOEFL Score', 'University Rating', 'SOP', 'LOR', 'CGPA', 'Research', 'Chance of Admit']

Tipos al releer:
GRE Score              Int64
TOEFL Score            Int64
University Rating      Int64
SOP                  float64
LOR                  float64
CGPA                 float64
Research               Int64
Chance of Admit      float64
dtype: object

Nulos al releer:
GRE Score            11
TOEFL Score          19
University Rating     6
SOP                  16
LOR                   8
CGPA                  2
Research             29
Chance of Admit       0
dtype: int64

Validación OK: dimensiones, columnas, tipos y nulos coinciden con el DataFrame intermedio.


## 📊 Análisis de resultados y conclusiones

- El RAW tiene **623 filas y 8 columnas** (7 features + el target `Chance of Admit`).
- La única representación de nulo en el RAW es la **cadena vacía** (`''`); no hay
  celdas con solo espacios ni otros marcadores (`NA`, `null`, etc.).
- **7 de 8 columnas** presentan valores faltantes (GRE 11, TOEFL 19,
  University Rating 6, SOP 16, LOR 8, CGPA 2, Research 29; `Chance of Admit` 0).
  Los nulos se normalizaron a `NaN`/`pd.NA` **sin imputar ni eliminar registros**.
- Tipos corregidos: 4 columnas enteras (`Int64`) y 4 continuas (`float64`).
- Nombres normalizados: se quitaron los espacios finales de `LOR ` y
  `Chance of Admit `.
- El resultado tipado se guardó en `data/02_intermediate/Admission_Predict.parquet`
  y la validación de lectura confirmó dimensiones, columnas, tipos y nulos.


## 💡 Propuestas e ideas

- Realizar el análisis exploratorio (EDA) univariable, bivariable y
  multivariable en una etapa posterior.
- Definir posteriormente una estrategia de tratamiento de los valores
  faltantes, sustentada en el análisis exploratorio.


## 📖 Referencias

- Zapata, J. R. — *Exploración inicial de datos* (referencia del curso):
  <https://joserzapata.github.io/post/ciencia-datos-proyecto-python/2-exploration/>
- `data/01_raw/Informacion.txt` — descripción del dataset y sus variables.
- `AGENTS.md` — reglas del proyecto (datos RAW, notebooks, capas de datos).
- `data/README.md` — convención de capas de datos (raw/intermediate/...).
- Documentación de pandas (dtypes nullable) y PyArrow (esquema en Parquet).
